# 01 — openFDA Data Access

Goal: query the official openFDA Drug Event API and build the first descriptive report counts.


In [1]:
import requests
import pandas as pd

BASE_URL = "https://api.fda.gov/drug/event.json"


In [2]:
DRUGS = ["SEMAGLUTIDE", "LIRAGLUTIDE", "DULAGLUTIDE"]
EVENTS = ["PANCREATITIS", "CHOLELITHIASIS", "CHOLECYSTITIS", "GASTROPARESIS", "NAUSEA", "VOMITING"]


## Helper function
Returns the total number of reports matching an openFDA search query.


In [3]:
def get_total(search_query):
    params = {"search": search_query, "limit": 1}
    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    return response.json()["meta"]["results"]["total"]


## First drug query
We will run and validate this together before writing the loop.


In [4]:
semaglutide_query = 'patient.drug.openfda.generic_name:"SEMAGLUTIDE"'
semaglutide_reports = get_total(semaglutide_query)
semaglutide_reports


73001

## First Drug × Event query


In [5]:
drug = "SEMAGLUTIDE"
event = "PANCREATITIS"

drug_event_query = (
    f'patient.drug.openfda.generic_name:"{drug}" '
    f'AND patient.reaction.reactionmeddrapt:"{event}"'
)
drug_event_query


'patient.drug.openfda.generic_name:"SEMAGLUTIDE" AND patient.reaction.reactionmeddrapt:"PANCREATITIS"'

In [6]:
drug_event_reports = get_total(drug_event_query)
drug_event_reports

1755

## Next session
After validating the first API calls, we will loop over all Drug × Event pairs, then construct 2×2 tables and calculate RORs with 95% CIs.

> Reporting signal ≠ causal effect.


In [7]:
rows = []

for drug in DRUGS:
    for event in EVENTS:
        query = (
            f'patient.drug.openfda.generic_name:"{drug}" '
            f'AND patient.reaction.reactionmeddrapt:"{event}"'
        )

        count = get_total(query)

        rows.append({
            "drug": drug,
            "event": event,
            "matching_reports": count
        })

summary = pd.DataFrame(rows)
summary

,drug,event,matching_reports
0,SEMAGLUTIDE,PANCREATITIS,1755
1,SEMAGLUTIDE,CHOLELITHIASIS,570
2,SEMAGLUTIDE,CHOLECYSTITIS,412
3,SEMAGLUTIDE,GASTROPARESIS,6
4,SEMAGLUTIDE,NAUSEA,10677
5,SEMAGLUTIDE,VOMITING,7411
6,LIRAGLUTIDE,PANCREATITIS,3237
7,LIRAGLUTIDE,CHOLELITHIASIS,449
8,LIRAGLUTIDE,CHOLECYSTITIS,226
9,LIRAGLUTIDE,GASTROPARESIS,9


In [8]:
drug_totals = {}

for drug in DRUGS:
    query = f'patient.drug.openfda.generic_name:"{drug}"'
    drug_totals[drug] = get_total(query)

drug_totals

{'SEMAGLUTIDE': 73001, 'LIRAGLUTIDE': 50704, 'DULAGLUTIDE': 88160}

In [9]:
event_totals = {}

for event in EVENTS:
    query = f'patient.reaction.reactionmeddrapt:"{event}"'
    event_totals[event] = get_total(query)

event_totals

{'PANCREATITIS': 80791,
 'CHOLELITHIASIS': 32537,
 'CHOLECYSTITIS': 26359,
 'GASTROPARESIS': 378,
 'NAUSEA': 779375,
 'VOMITING': 467918}

In [10]:
params = {"limit": 1}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=30
)

response.raise_for_status()

total_reports = response.json()["meta"]["results"]["total"]

total_reports

20692690

In [11]:
a = 1755
b = 73001 - a
c = 80791 - a
d = total_reports - a - b - c

a, b, c, d

(1755, 71246, 79036, 20540653)

In [12]:
ror = (a * d) / (b * c)
ror

6.40185639117807

In [13]:
import math

se_log_ror = math.sqrt(
    (1 / a) +
    (1 / b) +
    (1 / c) +
    (1 / d)
)

lower_ci = math.exp(math.log(ror) - 1.96 * se_log_ror)
upper_ci = math.exp(math.log(ror) + 1.96 * se_log_ror)

ror, lower_ci, upper_ci

(6.40185639117807, 6.102611144592764, 6.7157753103081586)

In [14]:
def calculate_ror(row):
    drug = row["drug"]
    event = row["event"]

    a = row["matching_reports"]
    b = drug_totals[drug] - a
    c = event_totals[event] - a
    d = total_reports - a - b - c

    ror = (a * d) / (b * c)

    se_log_ror = math.sqrt(
        (1 / a) +
        (1 / b) +
        (1 / c) +
        (1 / d)
    )

    lower_ci = math.exp(math.log(ror) - 1.96 * se_log_ror)
    upper_ci = math.exp(math.log(ror) + 1.96 * se_log_ror)

    return pd.Series({
        "ROR": ror,
        "CI_lower": lower_ci,
        "CI_upper": upper_ci
    })

In [15]:
results = summary.copy()

ror_stats = results.apply(
    calculate_ror,
    axis=1
)

results = pd.concat(
    [results, ror_stats],
    axis=1
)

results

,drug,event,matching_reports,ROR,CI_lower,CI_upper
0,SEMAGLUTIDE,PANCREATITIS,1755,6.401856,6.102611,6.715775
1,SEMAGLUTIDE,CHOLELITHIASIS,570,5.068236,4.663881,5.507647
2,SEMAGLUTIDE,CHOLECYSTITIS,412,4.504790,4.085903,4.966620
3,SEMAGLUTIDE,GASTROPARESIS,6,4.556060,2.033649,10.207113
4,SEMAGLUTIDE,NAUSEA,10677,4.424053,4.333612,4.516383
5,SEMAGLUTIDE,VOMITING,7411,4.946248,4.828002,5.067389
6,LIRAGLUTIDE,PANCREATITIS,3237,18.082706,17.438137,18.751101
7,LIRAGLUTIDE,CHOLELITHIASIS,449,5.738525,5.226013,6.301298
8,LIRAGLUTIDE,CHOLECYSTITIS,226,3.531981,3.097601,4.027274
9,LIRAGLUTIDE,GASTROPARESIS,9,9.931040,5.126150,19.239692


In [16]:
results["ROR"] = results["ROR"].round(2)
results["CI_lower"] = results["CI_lower"].round(2)
results["CI_upper"] = results["CI_upper"].round(2)

results

,drug,event,matching_reports,ROR,CI_lower,CI_upper
0,SEMAGLUTIDE,PANCREATITIS,1755,6.40,6.10,6.72
1,SEMAGLUTIDE,CHOLELITHIASIS,570,5.07,4.66,5.51
2,SEMAGLUTIDE,CHOLECYSTITIS,412,4.50,4.09,4.97
3,SEMAGLUTIDE,GASTROPARESIS,6,4.56,2.03,10.21
4,SEMAGLUTIDE,NAUSEA,10677,4.42,4.33,4.52
5,SEMAGLUTIDE,VOMITING,7411,4.95,4.83,5.07
6,LIRAGLUTIDE,PANCREATITIS,3237,18.08,17.44,18.75
7,LIRAGLUTIDE,CHOLELITHIASIS,449,5.74,5.23,6.30
8,LIRAGLUTIDE,CHOLECYSTITIS,226,3.53,3.10,4.03
9,LIRAGLUTIDE,GASTROPARESIS,9,9.93,5.13,19.24
